# Uplift-анализ маркетинговой рассылки

**Бизнес-вопрос:** кому отправить письмо с мужскими товарами, чтобы получить дополнительные покупки при ограниченном бюджете?

Мы сравним две модели и проверим их на отложенной выборке.

In [ ]:
from pathlib import Path
import gzip
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import roc_auc_score

DATA = Path('../data/raw/hillstrom.csv.gz')
df = pd.read_csv(DATA, compression='gzip')
df.head()

## 1. Подготовка данных

Для честного uplift-анализа оставляем только контроль и письмо с мужскими товарами. Признаки описывают клиента **до** рассылки.

In [ ]:
df = df[df['segment'].isin(['No E-Mail', 'Mens E-Mail'])].copy()
df['treatment'] = (df['segment'] == 'Mens E-Mail').astype(int)
df['conversion'] = df['conversion'].astype(int)

features = ['recency', 'history', 'mens', 'womens', 'zip_code', 'channel', 'newbie']
X = df[features].copy()
y = df['conversion']
t = df['treatment']
print(df['segment'].value_counts())
print('Конверсия без письма:', round(y[t == 0].mean(), 4))
print('Конверсия с письмом:', round(y[t == 1].mean(), 4))

## 2. Разделение на обучение и тест

Используем одинаковый random seed и стратификацию по группе воздействия. Тестовая выборка не участвует в обучении.

In [ ]:
from sklearn.model_selection import train_test_split
idx_train, idx_test = train_test_split(np.arange(len(df)), test_size=0.3, random_state=42, stratify=t)
X_train, X_test = X.iloc[idx_train], X.iloc[idx_test]
y_train, y_test = y.iloc[idx_train], y.iloc[idx_test]
t_train, t_test = t.iloc[idx_train], t.iloc[idx_test]

## 3. Две uplift-модели

T-learner обучает отдельную модель результата для клиентов с письмом и без него. Разница двух прогнозов — оценка индивидуального uplift.

Сравним логистическую регрессию как интерпретируемую базовую модель и градиентный бустинг как более гибкую модель.

In [ ]:
cat = ['zip_code', 'channel']
num = ['recency', 'history', 'mens', 'womens', 'newbie']
pre = ColumnTransformer([('num', SimpleImputer(strategy='median'), num), ('cat', OneHotEncoder(handle_unknown='ignore'), cat)])

def make_logistic():
    return Pipeline([('pre', pre), ('model', LogisticRegression(max_iter=1000, C=0.5))])

def fit_tlearner(factory):
    m0, m1 = factory(), factory()
    m0.fit(X_train[t_train.values == 0], y_train[t_train.values == 0])
    m1.fit(X_train[t_train.values == 1], y_train[t_train.values == 1])
    return m0, m1

def predict_uplift(models, X_part):
    m0, m1 = models
    return m1.predict_proba(X_part)[:, 1] - m0.predict_proba(X_part)[:, 1]

log_models = fit_tlearner(make_logistic)
log_uplift = predict_uplift(log_models, X_test)
print('Медианный uplift логистической модели:', round(float(np.median(log_uplift)), 4))

In [ ]:
# Простая модель на числовых признаках: две модели градиентного бустинга
boost_features = ['recency', 'history', 'mens', 'womens', 'newbie']
def fit_boost():
    m0 = HistGradientBoostingRegressor(max_iter=120, learning_rate=0.05, max_leaf_nodes=12, random_state=42)
    m1 = HistGradientBoostingRegressor(max_iter=120, learning_rate=0.05, max_leaf_nodes=12, random_state=42)
    m0.fit(X_train.loc[t_train.values == 0, boost_features], y_train[t_train.values == 0])
    m1.fit(X_train.loc[t_train.values == 1, boost_features], y_train[t_train.values == 1])
    return m0, m1
boost_models = fit_boost()
boost_uplift = boost_models[1].predict(X_test[boost_features]) - boost_models[0].predict(X_test[boost_features])

## 4. Проверка стратегии при ограниченном бюджете

Ранжируем клиентов по uplift и отправляем письмо только 20% аудитории. Сравнение делаем с простым правилом: отправить письмо случайным 20% клиентов.

In [ ]:
def incremental_purchases(scores, budget=0.2):
    n = len(scores); k = int(n * budget)
    chosen = np.argsort(scores)[-k:]
    treated = t_test.to_numpy()[chosen]
    outcomes = y_test.to_numpy()[chosen]
    # Оценка эффекта через разницу конверсий в тестовой выборке
    return int(k), float(outcomes.mean()), int(outcomes.sum())

rows=[]
for name, scores in [('Логистическая регрессия', log_uplift), ('Градиентный бустинг', boost_uplift)]:
    k, rate, purchases = incremental_purchases(scores)
    rows.append({'Модель': name, 'Клиентов в рассылке': k, 'Конверсия выбранных': rate, 'Покупок': purchases})
result = pd.DataFrame(rows)
result

## Вывод

Модель нужно выбирать не по красивому ROC-AUC, а по дополнительным покупкам при реальном бюджете. В рабочем процессе я бы проверил выбранную стратегию на новом A/B-тесте и сравнил её со случайной рассылкой. Клиентам с высокой базовой вероятностью покупки письмо может быть не нужно: uplift ищет тех, чьё поведение действительно меняется под воздействием предложения.